# Cropland Masking

By default, evy restricts EVI statistics to cropland pixels (`mask_cropland=True`). This recipe compares masked vs. unmasked results to show the effect of the cropland filter on the EVI signal.

**Important for paper methodology:** The GEE backend uses Dynamic World (class 4) for its cropland mask, while the local backend uses ESA WorldCover (class 40). These products may disagree at the pixel level. When writing up your methodology, state which backend you used. See [Design Decisions](../../docs/design-decisions.md) for details.

In [ ]:
import evy

In [ ]:
gdf = evy.get_boundaries("KEN", admin_level=1)
gdf[["shapeName"]].head()

## With cropland masking (default)

This is the default behavior — only cropland pixels contribute to the zonal statistics.

In [ ]:
df_masked = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    start_date="2023-01-01",
    end_date="2023-12-31",
    freq=evy.MONTHLY,
    stats=["mean"],
    mask_cropland=True,  # default, shown explicitly
)
df_masked.head()

## Without cropland masking

All land-cover types contribute — forest, grassland, urban, water, etc.

In [ ]:
df_unmasked = evy.zonal_stats(
    gdf,
    zone_col="shapeName",
    start_date="2023-01-01",
    end_date="2023-12-31",
    freq=evy.MONTHLY,
    stats=["mean"],
    mask_cropland=False,
)
df_unmasked.head()

## Side-by-side comparison

Let's plot both series to see how the mask affects the seasonal signal.

In [ ]:
evy.plot_time_series(df_masked, value_col="mean", title="EVI — Cropland Only")

In [ ]:
evy.plot_time_series(df_unmasked, value_col="mean", title="EVI — All Land Cover")

In regions with mixed land cover, the unmasked signal typically shows lower amplitude because non-agricultural vegetation (forest, grassland) has different seasonal dynamics than crops. The difference is most pronounced in regions where cropland is a small fraction of total land area.

If you need a **custom** land-cover mask beyond cropland, you can use the low-level pipeline (`load_modis` → `load_landcover` → `compute_zonal_stats`) to inspect and modify the raster directly. See the [quickstart](../quickstart.ipynb) for an example of the low-level pipeline.